# Transformation Equations:  LSST DP2 <--> SDSS DR18

_Douglas L. Tucker, Christina L. Adair, Meagan N. Porter_

_2026.06.16_

## 1. Import Modules

In [ ]:
import numpy as np
import pandas as pd

from lsst.daf.butler import Butler
import lsst.geom as geom

import pyvo

import os
import sys
import glob
import math
import datetime

from collections import OrderedDict as odict

from astropy.io import fits
from astropy.table import Table
from astropy.coordinates import SkyCoord
import astropy.units as u

import fitsio

from scipy import interpolate
from scipy.optimize import leastsq

import healpy as hp

import plotly
from plotly.offline import download_plotlyjs, plot, iplot
import plotly.graph_objs as go

import matplotlib.pyplot as plt

%matplotlib inline

## 2. Input Parameters

In [ ]:
# lsst-->SDSS DR18 (toLSST=False) or SDSS DR18-->LSST (toLSST=True)...
toLSST = True
#toLSST = False

if toLSST:
    # List of LSST bands on which to run the transform fit...
    #bandList = ['g_LSST', 'r_LSST', 'i_LSST', 'z_LSST', 'y_LSST']
    bandList = ['u_LSST', 'g_LSST', 'r_LSST', 'i_LSST', 'z_LSST', 'y_LSST', 'gi_LSST']
    # If SDSS-->LSST, then the mag and color to fit against are SDSS...
    # Dictionary of corresponding bands from the other survey...
    matchBand_dict = {'u_LSST':'u_sdss', 'g_LSST':'g_sdss', 'r_LSST':'r_sdss', 'i_LSST':'i_sdss', 'z_LSST':'z_sdss', 'y_LSST':'z_sdss', 'gi_LSST':'gi_sdss'}
    # Color to fit against...
    color_name_1_dict = {'u_LSST':'gi_sdss', 'g_LSST':'gi_sdss', 'r_LSST':'gi_sdss', 'i_LSST':'gi_sdss', 'z_LSST':'gi_sdss', 'y_LSST':'gi_sdss', 'gi_LSST':'gi_sdss'}
    # Name of color_name_1 as the label in the QA plots...
    colorLabel_1_dict = {'u_LSST':'$(g-i)_{sdss}$', 'g_LSST':'$(g-i)_{sdss}$', 'r_LSST':'$(g-i)_{sdss}$', 'i_LSST':'$(g-i)_{sdss}$', 'z_LSST':'$(g-i)_{sdss}$', 'y_LSST':'$(g-i)_{sdss}$', 'gi_LSST':'$(g-i)_{sdss}$'}
    # Color limits defining disjoint branches of the dmag vs. color plots
    #  (each branch will be fit separately)...
    color_limits_1_dict = {'u_LSST':[-5.,5.], 
                           'g_LSST':[-5.,5.], 
                           'r_LSST':[-5.,5.], 
                           'i_LSST':[-5.,5.],
                           'z_LSST':[-5.,5.],
                           'y_LSST':[-5.,5.],
                           'gi_LSST':[-5.,5.]
                          }
    
    
else:
    # List of SDSS bands on which to run the transform fit...
    bandList = ['u_sdss', 'g_sdss', 'r_sdss', 'i_sdss', 'z_sdss','gi_sdss']
    # If LSST-->SDSS, then the mag and color to fit against are LSST...
    # Dictionary of corresponding bands from the other survey...
    matchBand_dict = {'u_sdss':'u_LSST', 'g_sdss':'g_LSST', 'r_sdss':'r_LSST', 'i_sdss':'i_LSST', 'z_sdss':'z_LSST', 'gi_sdss':'gi_LSST'}
    # Color to fit against...
    color_name_1_dict = {'u_sdss':'gi_LSST', 'g_sdss':'gi_LSST', 'r_sdss':'gi_LSST', 'i_sdss':'gi_LSST', 'z_sdss':'gi_LSST', 'gi_sdss':'gi_LSST'}
    # Name of color_name_1 as the label in the QA plots...
    colorLabel_1_dict = {'u_sdss':'$(g-i)_{LSST}$', 'g_sdss':'$(g-i)_{LSST}$', 'r_sdss':'$(g-i)_{LSST}$', 'i_sdss':'$(g-i)_{LSST}$', 'z_sdss':'$(g-i)_{LSST}$', 'gi_sdss':'$(g-i)_{LSST}$'}
    # Color limits defining disjoint branches of the dmag vs. color plots
    #  (each branch will be fit separately)...    
    color_limits_1_dict = {'u_sdss':[-5.,5.], 
                           'g_sdss':[-5.,5.], 
                           'r_sdss':[-5.,5.], 
                           'i_sdss':[-5.,5.],
                           'z_sdss':[-5.,5.],
                           'gi_sdss':[-5.,5.]
                          }


# Order of polynomial fits...
norder = 1

# Sigma-clipping parameters...
nsigma = 3.0
niter = 3

# LSST data
# possible DP2 collections
#collections = 'LSSTCam/runs/DRP/20250515-20251214/v30_0_0_rc2/DM-53697'
#collections = 'LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881'
#collections = 'LSSTCam/runs/DRP/v30_0_4/DM-54249' #this is the French data facility - no tract 8524 as of 3/20/26
#collection = 'LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2' #used recalibrated_star_ junk below to make this work....3/26/26
#collection = 'LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3'
collection = 'LSSTCam/runs/DRP/DP2'

repo = 'dp2_prep'
skymap = 'lsst_cells_v2'
instrument = 'LSSTCam'

## Name of the SDSS file (if it exists)
#sdssFile = '/home/d/dltucker/DATA/LSST_COSMOS_dtucker.csv'

# Name of the SDSS cache file (if it exists)
sdss_cache_file = "sdss_by_tract_dp2.pkl"

# Use SDSS cache file?
useSDSSCacheFile = False

#use match file?
useMatchFile = False

# Name of the match file
#matchFile = '/home/d/dltucker/DATA/match.lsst_stars_all.w_2025_10.DM-49359a.SDSS_DR18.csv'
#matchFile = '/home/d/dltucker/DATA/match.LSST_stars_all.DP1.SDSS_DR18.csv'
#matchFile = 'match.LSST_COSMOS.DM-51580.SDSS_DR18.csv'
#matchFile = 'match_LSST_DP2_DM_53881_stage3_SDSS_DR18.csv'
matchFile = 'match_LSST_DP2_SDSS_DR18.csv'


# Base name of fit results output file...
#if toLSST:
#    resultsFileBaseName = 'transFit.PS1DR2_to_LSST'
#else:
#    resultsFileBaseName = 'transFit.LSST_to_PS1DR2'

# Base name of QA plot output files...
if toLSST:
    qaFileBaseName = 'qaPlot.SDSS_to_LSST.fit'
else:
    qaFileBaseName = 'qaPlot.LSST_to_SDSS.fit'

# Verbosity level (0, 1, 2, 3, ...)
verbose = 2


# COSMOS field
#tract_list = [9813]
#tract_dict={9813: 'COSMOS'}

In [ ]:
#tract_list = [7779, 7780, 7781, 7938, 7939, 7940, 7941, 7942, 7943, 7944, 7950, 7973, 7974, 7975, 7976, 7977, 7978, 7979, 7980, 7981, 7982, 7983, 7984, 7985, 7986, 7987, 7988, 7989, 7990, 7991, 7992, 7993, 7994, 7995, 7996, 7997, 7998, 7999, 8000, 8001, 8002, 8003, 8005, 8006, 8007, 8008, 8009, 8010, 8011, 8017, 8018, 8019, 8180, 8181, 8182, 8259, 8500, 8742, 8984, 9226, 9327, 9469, 9569, 9570, 9571, 9572, 9712, 9811, 9812, 9813, 9814, 9954, 10054, 10055, 10056]

## 3. Define Useful Functions

In [ ]:
# Useful class to stop "Run All" at a cell 
#  containing the command "raise StopExecution"
class StopExecution(Exception):
    def _render_traceback_(self):
        pass

In [ ]:
# No longer used in this notebook (2026.06.16)
def cross_match_catalogs(df1, df2, ra_name_1, dec_name_1, ra_name_2, dec_name_2):

    # Create SkyCoord objects for both dataframes
    coords1 = SkyCoord(ra=df1[ra_name_1].values*u.degree, 
                       dec=df1[dec_name_1].values*u.degree)

    coords2 = SkyCoord(ra=df2[ra_name_2].values*u.degree, 
                       dec=df2[dec_name_2].values*u.degree)

    # Match coordinates
    max_sep = 3 * u.arcsec  # Maximum separation
    idx, d2d, d3d = coords1.match_to_catalog_sky(coords2)

    # Create mask for matches within max_sep
    mask = d2d < max_sep

    # Additional mask to ensure indices are valid
    valid_idx_mask = idx[mask] < len(df2)
    combined_mask = mask.copy()
    combined_mask[mask] = valid_idx_mask
    
    # Create a new dataframe with matches
    matches = df1[combined_mask].copy()
    matches['match_idx'] = idx[combined_mask]  # Index of matching object in df2
    matches['separation_arcsec'] = d2d[combined_mask].arcsec  # Separation in arcseconds

    # Add columns from df2 for the matches
    for col in df2.columns:
        #matches[f'match_{col}'] = df2.loc[idx[mask], col].values
        # This is a safer way to to this, avoid out-of-bound indices:
        matches[f'match_{col}'] = df2.iloc[idx[combined_mask]][col].values

    # If multiple matches exist for the same source in df1, keep only the closest one
    matches = matches.loc[matches.groupby(matches.index)['separation_arcsec'].idxmin()]

    # If you want to see which objects in df1 had no matches:
    unmatched = df1[~combined_mask]

    return matches, unmatched

In [ ]:
def transform1ColorQAPlots1a(dmag, color1, res, norder, title, plotText, dmagName, colorLabel1, rms, outputFileName):

    # Prepare QA plots...
    #fig = plt.figure(figsize=(10,5))
    #fig = plt.figure(figsize=(40,20))
    fig = plt.figure(figsize=(20,10))
    fig.subplots_adjust(hspace=0.3)
    #fig.suptitle("This is a supertitle!")
    plt.rcParams.update({'font.size': 24})
    
    # We will exclude the lowest and highets 0.01% of color1, color2, 
    #  dmag, and residuals when plotting the QA figures...
    color1_desc = color1.describe(percentiles=[0.0001, 0.001, 0.01, 0.99, 0.999, 0.9999])
    dmag_desc = dmag.describe(percentiles=[0.0001, 0.001, 0.01, 0.99, 0.999, 0.9999])
    #res_desc = df.res.describe(percentiles=[0.0001, 0.001, 0.01, 0.99, 0.999, 0.9999])
    res_desc = res.describe(percentiles=[0.0001, 0.001, 0.01, 0.99, 0.999, 0.9999])
    #color1_min = color1_desc['1%']
    #color1_max = color1_desc['99%']
    #color1_min = color1_desc['min']
    #color1_max = color1_desc['max']
    #dmag_min = dmag_desc['1%']
    #dmag_max = dmag_desc['99%']
    #res_min = res_desc['1%']
    #res_max = res_desc['99%']
    color1_min = color1_desc['0.01%']
    color1_max = color1_desc['99.99%']
    dmag_min = dmag_desc['0.01%']
    dmag_max = dmag_desc['99.99%']
    res_min = res_desc['0.01%']
    res_max = res_desc['99.99%']
    # What the heck; let's just set this to -0.10 mag --> +0.10 mag...
    #res_min = -0.10
    #res_max = +0.10

    
    # Plot 1:  Descriptive text...
    #plt.subplot(231)
    plt.subplot(221)
    plt.text(0.1,0.80,title,fontsize=24)
    plt.text(0.00,0.40,plot1Text,fontsize=12)
    plt.axis('off')

    
    # Plot 2:  2D hexbin histogram of dmag vs. color1...
    #plt.subplot(232) 
    plt.subplot(222)
    if len(dmag) < 5000:
        plt.scatter(color1, dmag, alpha=0.75)
        #hb=plt.hexbin(color1, dmag, gridsize=100, cmap='inferno_r')
    else:
        hb=plt.hexbin(color1, dmag, gridsize=100, bins='log', cmap='inferno')
    plt.axis([color1_min, color1_max, dmag_min, dmag_max])
    plt.xlabel(colorLabel1)
    plt.ylabel(dmagName)
    if len(dmag) >= 10000:
        cb = fig.colorbar(hb)
        cb.set_label('log10(N)')
    plt.grid(color='blue')
    plt.grid(True)


    # Plot 3:  1d histogram of residuals...
    #plt.subplot(234) 
    plt.subplot(223) 
    #plt.hist(df.loc[:,'res'],bins=100)
    if len(res) < 100:
        plt.hist(res,bins=10)
    else:
        plt.hist(res,bins=100)
    plt.xlabel('residuals [mag]')
    plt.ylabel('Number')
    plt.grid(True)
    plt.grid(color='blue')

    
    # Plot 4:  2d hexbin histogram of residuals vs. color1...
    #plt.subplot(235) 
    plt.subplot(224)
    if len(res) < 5000:
        plt.scatter(color1, res, alpha=0.75)
        #hb = plt.hexbin(color1, res, gridsize=100, cmap='inferno_r')
    else:
        hb = plt.hexbin(color1, res, gridsize=100, bins='log', cmap='inferno')
    plt.axis([color1_min, color1_max, res_min, res_max])
    plt.xlabel(colorLabel1)
    plt.ylabel('residuals [mag]')
    if len(res) >= 10000:
        cb = plt.colorbar(hb)
        cb.set_label('log10(N)')
    plt.grid(True)
    plt.grid(color='blue')

    
    # Plot...
    plt.tight_layout()
    #plt.show()
    plt.savefig(outputFileName)

    return 0


##################################


In [ ]:
# Kudos to Claude-3.5-Sonnet for improving on old outlier rejection code...

def poly_fit_with_sigma_clip(x, y, degree=1, sigma=3.0, maxiters=5):
    """
    Perform polynomial fit with iterative sigma clipping
    
    Parameters:
    -----------
    x : array-like
        Independent variable
    y : array-like 
        Dependent variable
    degree : int
        Degree of polynomial fit
    sigma : float
        Sigma clipping threshold
    maxiters : int
        Maximum number of sigma clipping iterations
        
    Returns:
    --------
    coeffs : array
        Polynomial coefficients
    mask : array
        Boolean mask indicating non-clipped points
    rms : float
        RMS of residuals
    """

    # Import relevant modules
    import numpy as np
    from astropy.stats import sigma_clip
    
    # Initial fit using all points
    x = np.asarray(x)
    y = np.asarray(y)
    mask = np.ones_like(x, dtype=bool)
    
    for _ in range(maxiters):
        print(len(x[mask]), len(y[mask]), len(mask))

        # Fit polynomial to non-masked points
        coeffs, cov = np.polyfit(x[mask], y[mask], degree, cov=True)
        
        # Calculate residuals
        yfit = np.polyval(coeffs, x)
        residuals = y - yfit
        
        # Update mask with sigma clipping
        new_mask = ~sigma_clip(residuals, sigma=sigma).mask
        
        # Check for convergence
        if np.array_equal(mask, new_mask):
            break
        
        mask = new_mask
    
    # Calculate final RMS
    final_residuals = y[mask] - np.polyval(coeffs, x[mask])
    rms = np.sqrt(np.mean(final_residuals**2))

    print(len(x[mask]), len(y[mask]), len(mask))

    # Calculate coefficient errors from diagonal of covariance matrix
    coeff_errors = np.sqrt(np.diag(cov))
        
    return coeffs, coeff_errors, x[mask], y[mask], final_residuals, rms

In [ ]:
# Kudos to Copilot ChatGPT-5 for these functions for generating Markdown output 
#  of the fit equations...
# See:  https://copilot.microsoft.com/shares/Nopgin8hutEqEmjqJkteY

def latexify_label(label: str) -> str:
    """Strip $ and convert underscores to LaTeX subscripts."""
    label = label.strip('$')
    return re.sub(r'_(\w+)', r'_{\1}', label)

def latexify_name(name: str) -> str:
    """Convert underscores in names to LaTeX subscripts."""
    return re.sub(r'_(\w+)', r'_{\1}', name)

def make_eq_str(dmagName, p_branch, colorLabel_1, norder):
    """
    Build a LaTeX-formatted transformation equation for Markdown.
    Ensures all terms are inside a single $...$ block with proper signs.
    """
    label = latexify_label(colorLabel_1)           # e.g., (g-i)_{LSST}
    dmagName_math = latexify_name(dmagName)        # e.g., U - u_{LSST}

    terms = []
    power = norder
    for coeff in p_branch:
        if power == 0:
            # Constant term: no variable
            terms.append(f"{coeff:+.3f}")
        elif power == 1:
            # Linear term
            terms.append(f"{coeff:+.3f} {label}")
        else:
            terms.append(f"{coeff:+.3f} {label}^{power}")
            # Higher-order term
        power -= 1

    # Join all terms with spaces; signs are already included in coeff formatting
    eq_body = " ".join(terms)
    return f"${dmagName_math} = {eq_body}$"

def make_range_str(color_limits, colorLabel_1, ibranch):
    label = latexify_label(colorLabel_1)
    # Use LaTeX \leq for "less than or equal to"
    return (f"${color_limits[ibranch]:.1f} < {label} \\leq "
            f"{color_limits[ibranch+1]:.1f}$")

def make_output_filename(base, magName, magName_match, color_name_1, norder):
    return f"{base}.dmag_{magName}-{magName_match}.{color_name_1}.norder{norder}.qa1.png"

In [ ]:
# Kudos to Copilot ChatGPT-5 for improving the plot text generation

def make_plot_text(dmagName, p_branch, colorLabel_1, norder,
                   color_limits, ibranch, stddev_branch):
    """
    Build a descriptive text string for plots, valid for any polynomial order.
    Example output:
    "g - r = -0.041 -0.302*(g-i)   [0.2 < (g-i) <= 2.5]   [rms: 0.022]"
    """
    terms = []
    power = norder
    for coeff in p_branch:
        if power == 0:
            # Constant term, but still include sign
            terms.append(f"{coeff:+.3f}")
        elif power == 1:
            terms.append(f"{coeff:+.3f}*{colorLabel_1}")
        else:
            terms.append(f"{coeff:+.3f}*{colorLabel_1}^{power}")
        power -= 1

    poly_str = " ".join(terms)

    lo = color_limits[ibranch]
    hi = color_limits[ibranch+1]
    range_str = f"[{lo:.1f} < {colorLabel_1} <= {hi:.1f}]"
    rms_str = f"[rms: {stddev_branch:.3f}]"

    return f"{dmagName} = {poly_str}  {range_str}  {rms_str}"

## X. Read in Matched Catalog

Only used if `useMatchFile == True` (i.e., to save time not querying LSST DP2/SDSS DR18 and matching the 2 catalogs)...  



In [ ]:
## Check to make sure matchFile exists...
if useMatchFile:
    if os.path.isfile(matchFile)==False:
        print("""ERROR:  matchFile %s does not exist...""" % (matchFile))
    if verbose > 0:
        print('matchFile: ', matchFile)


In [ ]:
if useMatchFile:
    tab = Table.read(matchFile, format='csv')
    display(tab)

In [ ]:
if useMatchFile:
    matches = tab.to_pandas()
    display(matches)

## 4. Query LSST Catalog

In [ ]:
# ============================================================
# 4. Overlapping tracts + LSST DP2 stars (OPTIMIZED)
# ============================================================

if not useMatchFile:

    import numpy as np
    import pandas as pd
    from shapely.geometry import Polygon, Point
    from lsst.daf.butler import Butler
    from tqdm.notebook import tqdm

    # --- 4.1 SDSS DR18 imaging footprint polygon ---
    sdss_polygon_coords = np.array([
        [0, -10], [60, -10], [60, 10], [120, 10], [120, 60],
        [240, 60], [240, 10], [300, 10], [300, -10], [360, -10]
    ])
    sdss_poly = Polygon(sdss_polygon_coords)

    def sample_polygon(poly, n_ra=200, n_dec=200):
        ra_min, dec_min, ra_max, dec_max = poly.bounds
        ras = np.linspace(ra_min, ra_max, n_ra)
        decs = np.linspace(dec_min, dec_max, n_dec)
        pts = []
        for ra in ras:
            for dec in decs:
                if poly.contains(Point(ra, dec)):
                    pts.append((ra, dec))
        return np.array(pts)

    def tracts_overlapping_sdss(skymap, sdss_poly):
        pts = sample_polygon(sdss_poly, n_ra=200, n_dec=200)
        ra = pts[:, 0]
        dec = pts[:, 1]
        tract_ids = skymap.findTractIdArray(ra, dec, degrees=True)
        return sorted(set(tract_ids))

    # --- 4.2 Build initial tract list from geometry only ---
    skybutler = Butler(repo, collections=collection, skymap=skymap)
    skymap_obj = skybutler.get("skyMap", skymap="lsst_cells_v2")

    geom_tract_list = tracts_overlapping_sdss(skymap_obj, sdss_poly)
    if verbose > 2:  print("Geometric SDSS-overlap tracts:", geom_tract_list)

    # --- 4.3 Helper: tract vertices in RA/Dec ---
    def tract_vertices(tractId):
        tract = skymap_obj.generateTract(tractId)
        verts = tract.getVertexList()
        return [(v.getRa().asDegrees(), v.getDec().asDegrees()) for v in verts]

    # --- 4.4 Query LSST DP2 only where data exist ---
    butler = Butler(repo, collections=collection)
    INCOLS = ["coord_ra", "coord_dec", "tract", "patch"]
    for band in "ugrizy":
        INCOLS += [
            f"{band}_psfFlux", f"{band}_psfFluxErr",
            f"{band}_ap12Flux", f"{band}_ap12FluxErr",
            f"{band}_extendedness", f"{band}_psfFlux_flag"
        ]

    LSST_stars_list = []
    tracts_with_rubin = []

    #for tractId in geom_tract_list:
    #    print(f"Processing LSST tract {tractId},")
    # tqdm progress bar
    for tractId in tqdm(geom_tract_list, desc="Processing LSST tracts"):
        try:
            raw = butler.get(
                "object",
                dataId={"skymap": "lsst_cells_v2", "tract": tractId},
                collections=[collection],
                parameters={"columns": INCOLS}
            ).to_pandas()
        except Exception as e:
            if verbose > 2:  print(f"  No DP2 object table for tract {tractId}: {e}")
            continue

        raw.insert(0, "tractId", tractId)

        # SNR + flag cuts
        sel = (raw["r_psfFlux"] / raw["r_psfFluxErr"] > 5)
        for b in ["g", "r", "i"]:
            sel &= (raw[f"{b}_psfFlux_flag"] == 0)

        LSST = raw[sel]

        # Star selection
        stars = LSST[(LSST["g_extendedness"] < 0.5) &
                     (LSST["r_extendedness"] < 0.5)]

        if verbose > 2:  print(f"  Stars: {len(stars)}")
        if len(stars) == 0:
            continue

        LSST_stars_list.append(stars)
        tracts_with_rubin.append(tractId)

    # --- 4.5 Final LSST star catalog + tract list ---
    if len(LSST_stars_list) == 0:
        raise RuntimeError("No LSST stars found in any overlapping tract.")

    LSST_stars_all = pd.concat(LSST_stars_list, ignore_index=True)
    tract_list = sorted(set(tracts_with_rubin))

    print(f"\nTotal LSST stars: {len(LSST_stars_all)}")
    print("Tracts with BOTH SDSS footprint overlap AND Rubin DP2 data:")
    print(tract_list)

    # --- 4.6 table_patches only for tracts with Rubin data ---
    table_patches = pd.DataFrame({
        "tractId": tract_list,
        "vertices_deg": [tract_vertices(t) for t in tract_list]
    })


In [ ]:
if not useMatchFile:
    display(LSST_stars_all)


## 5. Query SDSS DR18 Catalog

In [ ]:
# ============================================================
# 5. Query SDSS DR18 (OPTIMIZED + CACHED + PANDAS QA FILTERS)
# ============================================================

if not useMatchFile:

    from astroquery.sdss import SDSS
    import time
    import os
    import pickle

    # ------------------------------------------------------------
    # 5.0 SDSS QA mask function (pandas implementation)
    # ------------------------------------------------------------
    def clean_sdss_photometry(df):
        """DR18-compatible SDSS photometric quality cuts."""

        PSF_MEAS = 0x10000000

        # Require PSF measurements in g,r,i (u,z optional)
        psf_meas = (
            ((df.flags_g & PSF_MEAS) != 0) &
            ((df.flags_r & PSF_MEAS) != 0) &
            ((df.flags_i & PSF_MEAS) != 0)
        )

        # Primary detection
        primary = (df.mode != 0)
        print()

        # SDSS clean flag (already includes many QA checks)
        clean_phot = (df.clean == 1)

        # Reasonable magnitude errors
        good_err = (
            (df.g_err < 0.2) &
            (df.r_err < 0.2) &
            (df.i_err < 0.2)
        )

        # u-band special handling (looser)
        u_ok = (df.u_err < 0.5)

        #mask = psf_meas & primary & clean_phot & good_err & u_ok
        mask = psf_meas & clean_phot & good_err & u_ok
        return df[mask].reset_index(drop=True)


    # ------------------------------------------------------------
    # 5.1 Helper: tract bounding box
    # ------------------------------------------------------------
    def tract_bounding_box(vertices):
        ras = np.array([v[0] for v in vertices])
        decs = np.array([v[1] for v in vertices])
        ras_wrapped = np.unwrap(np.deg2rad(ras))
        ras_deg = np.rad2deg(ras_wrapped)
        return ras_deg.min(), ras_deg.max(), decs.min(), decs.max()


    # ------------------------------------------------------------
    # 5.2 SDSS query (lightweight RA/Dec window only)
    # ------------------------------------------------------------
    def query_sdss_dr18_box(ra_min, ra_max, dec_min, dec_max, max_retries=5):

        query = f"""
            SELECT s.ra, s.dec,
                s.psfMag_u AS u, s.psfMag_g AS g, s.psfMag_r AS r,
                s.psfMag_i AS i, s.psfMag_z AS z,
                s.psfMagErr_u as u_err, s.psfMagErr_g as g_err, s.psfMagErr_r as r_err,
                s.psfMagErr_i as i_err, s.psfMagErr_z as z_err,
                s.extinction_u, s.extinction_g, s.extinction_r,
                s.extinction_i, s.extinction_z,
                s.flags_u, s.flags_g, s.flags_r, s.flags_i, s.flags_z,
                s.mode, s.clean
            FROM Star s
            WHERE s.ra BETWEEN {ra_min} AND {ra_max}
              AND s.dec BETWEEN {dec_min} AND {dec_max}
        """

        for attempt in range(max_retries):
            try:
                result = SDSS.query_sql(
                    query,
                    data_release=18,
                    timeout=120,
                    url="https://skyserver.sdss.org/dr18/SkyServerWS/SearchTools/SqlSearch"
                )
                if result is None:
                    return None
                return result.to_pandas()
            except Exception as e:
                print(f"  SDSS query failed ({attempt+1}/{max_retries}): {e}")
                time.sleep(2 + 2*attempt)

        print("  SDSS query failed after all retries.")
        return None


    # ------------------------------------------------------------
    # 5.3 Load from cache or query SDSS
    # ------------------------------------------------------------
    if useSDSSCacheFile and os.path.isfile(sdss_cache_file):
        print(f"Loading SDSS results from cache: {sdss_cache_file}")
        with open(sdss_cache_file, "rb") as f:
            sdss_by_tract = pickle.load(f)

    else:
        sdss_by_tract = {}

        for tractId, vertices in zip(table_patches["tractId"],
                                     table_patches["vertices_deg"]):

            print(f"\n=== SDSS query for tract {tractId} ===")
            ra_min, ra_max, dec_min, dec_max = tract_bounding_box(vertices)
            print(f"  RA [{ra_min:.3f}, {ra_max:.3f}], Dec [{dec_min:.3f}, {dec_max:.3f}]")

            sdss_df = query_sdss_dr18_box(ra_min, ra_max, dec_min, dec_max)

            if sdss_df is None or len(sdss_df) == 0:
                print("  No SDSS sources.")
                continue

            # Skip malformed tables
            if not {"ra", "dec"}.issubset(sdss_df.columns):
                print("  SDSS returned malformed table — skipping.")
                continue

            print(f"  Retrieved {len(sdss_df)} SDSS sources")

            # Apply SDSS QA cuts in pandas
            sdss_clean = clean_sdss_photometry(sdss_df)
            print(f"  After SDSS QA cuts: {len(sdss_clean)} sources")

            if len(sdss_clean) == 0:
                print("  No clean SDSS sources after QA cuts.")
                continue

            sdss_by_tract[tractId] = sdss_clean.copy()

        print("\nSDSS DR18 querying complete.")
        print(f"Saving SDSS results to cache: {sdss_cache_file}")
        with open(sdss_cache_file, "wb") as f:
            pickle.dump(sdss_by_tract, f)

    print(f"\nNumber of tracts with SDSS data: {len(sdss_by_tract)}")


In [ ]:
#print(sdss_df)
#print(sdss_clean)

## 6. Match LSST and SDSS DR18 stars

In [ ]:
# ============================================================
# 6. Cross‑match LSST DP2 ↔ SDSS DR18 (OPTIMIZED)
# ============================================================

if not useMatchFile:
    
    import astropy.units as u
    from astropy.coordinates import SkyCoord

    def crossmatch_rubin_sdss(rubin_df, sdss_df, max_sep_arcsec=1.0):
        rubin_coords = SkyCoord(
            rubin_df["coord_ra"].astype(float).values * u.deg,
            rubin_df["coord_dec"].astype(float).values * u.deg
        )
        sdss_coords = SkyCoord(
            sdss_df["ra"].astype(float).values * u.deg,
            sdss_df["dec"].astype(float).values * u.deg
        )
        idx, sep2d, _ = rubin_coords.match_to_catalog_sky(sdss_coords)
        mask = sep2d < (max_sep_arcsec * u.arcsec)

        matched_rubin = rubin_df[mask].copy()
        matched_rubin["sdss_index"] = idx[mask]
        matched_rubin["sdss_sep_arcsec"] = sep2d[mask].arcsec

        matched_sdss = sdss_df.iloc[idx[mask]].reset_index(drop=True)
        return matched_rubin.reset_index(drop=True), matched_sdss

    matched_rows = []

    for tractId, sdss_df in sdss_by_tract.items():
        rubin_df = LSST_stars_all[LSST_stars_all["tractId"] == tractId]

        if len(rubin_df) == 0:
            continue

        print(f"\nMatching tract {tractId}: Rubin {len(rubin_df)}, SDSS {len(sdss_df)}")

        #m_rubin, m_sdss = crossmatch_rubin_sdss(rubin_df, sdss_df)
        # Skip malformed SDSS tables
        required_cols = {"ra", "dec"}
        if not required_cols.issubset(sdss_df.columns):
            print(f"  Skipping tract {tractId}: SDSS table missing RA/Dec columns")
            continue

        m_rubin, m_sdss = crossmatch_rubin_sdss(rubin_df, sdss_df)


        print(f"  Matched {len(m_rubin)} objects")
        if len(m_rubin) > 0:
            merged = pd.concat(
                [m_rubin.reset_index(drop=True),
                 m_sdss.reset_index(drop=True)],
                axis=1
            )
            matched_rows.append(merged)

    if len(matched_rows) == 0:
        raise RuntimeError("No Rubin–SDSS matches found in any tract.")

    matches = pd.concat(matched_rows, ignore_index=True)
    print(f"\nTotal matched objects: {len(matches)}")


In [ ]:
if not useMatchFile:
    display(matches)

In [ ]:
# Save new match file...
if not useMatchFile:
    matches.to_csv(matchFile,index=False)

In [ ]:
# Plot Aitoff sky plot of new match file...

if not useMatchFile:

    # Extract RA/Dec from your final matches table
    ra = matches["coord_ra"].values
    dec = matches["coord_dec"].values

    # Convert RA to [-180, +180] degrees
    ra_wrapped = np.remainder(ra + 180.0, 360.0) - 180.0

    # Convert to radians
    ra_rad = np.deg2rad(ra_wrapped)
    dec_rad = np.deg2rad(dec)

    # Plot
    plt.figure(figsize=(10, 6))
    ax = plt.subplot(111, projection="aitoff")

    ax.scatter(ra_rad, dec_rad, s=1, alpha=0.5)
    ax.grid(True)

    plt.title("Aitoff Projection of Rubin–SDSS Matched Stars", pad=20)
    plt.show()


##  7. Add ABmag Columns to Matched Catalog Data Frame

In [ ]:
# Copy matches to df...
df = matches.copy()

In [ ]:
# Define common parameters
flux_bands = ['u', 'g', 'r', 'i', 'z', 'y']
offset = 31.4 # For magnitude calculation
sentinel_value = -9999.0

# Loop through each band to calculate both magnitude and magnitude error
for band in flux_bands:
    
    flux_col = f'{band}_psfFlux'
    flux_err_col = f'{band}_psfFluxErr'
    mag_col = f'{band}_psfMag'
    mag_err_col = f'{band}_psfMagErr'

    # Condition for valid flux (must be positive for log10 and division)
    valid_flux_condition = ((df[flux_col] > 0) & (df[flux_col].notna()))

    # Calculate magnitude
    df[mag_col] = np.where(valid_flux_condition,
                           -2.5 * np.log10(df[flux_col]) + offset,
                           sentinel_value)

    # Calculate magnitude error
    df[mag_err_col] = np.where(valid_flux_condition,
                               1.086 * df[flux_err_col] / df[flux_col],
                               sentinel_value)



In [ ]:
# Rename columns...
df.rename(columns={'coord_ra':'RA_LSST',
                   'coord_dec':'DEC_LSST',
                   'u_psfMag':'u_LSST',
                   'g_psfMag':'g_LSST',
                   'r_psfMag':'r_LSST',
                   'i_psfMag':'i_LSST',
                   'z_psfMag':'z_LSST',
                   'y_psfMag':'y_LSST',
                   'u_psfMagErr':'u_err_LSST',
                   'g_psfMagErr':'g_err_LSST',
                   'r_psfMagErr':'r_err_LSST',
                   'i_psfMagErr':'i_err_LSST',
                   'z_psfMagErr':'z_err_LSST',
                   'y_psfMagErr':'y_err_LSST',
                   'u':'u_sdss',
                   'g':'g_sdss',
                   'r':'r_sdss',
                   'i':'i_sdss',
                   'z':'z_sdss',
                   'u_err':'u_err_sdss',
                   'g_err':'g_err_sdss',
                   'r_err':'r_err_sdss',
                   'i_err':'i_err_sdss',
                   'z_err':'z_err_sdss',
                   'flags_u':'u_flags_sdss',
                   'flags_g':'g_flags_sdss',
                   'flags_r':'r_flags_sdss',
                   'flags_i':'i_flags_sdss',
                   'flags_z':'z_flags_sdss',
                   'mode':'mode_sdss',
                   'clean':'clean_sdss'
                 },inplace=True)

df.head(5)


## 8. Add Color Columns to Matched Catalog Data Frame

In [ ]:
# Add color columns...
df.loc[:,'ug_LSST'] = df.loc[:,'u_LSST'] - df.loc[:,'g_LSST']
df.loc[:,'gr_LSST'] = df.loc[:,'g_LSST'] - df.loc[:,'r_LSST']
df.loc[:,'ri_LSST'] = df.loc[:,'r_LSST'] - df.loc[:,'i_LSST']
df.loc[:,'iz_LSST'] = df.loc[:,'i_LSST'] - df.loc[:,'z_LSST']
df.loc[:,'zy_LSST'] = df.loc[:,'z_LSST'] - df.loc[:,'y_LSST']
df.loc[:,'gi_LSST'] = df.loc[:,'g_LSST'] - df.loc[:,'i_LSST']

df.loc[:,'ug_sdss'] = df.loc[:,'u_sdss'] - df.loc[:,'g_sdss']
df.loc[:,'gr_sdss'] = df.loc[:,'g_sdss'] - df.loc[:,'r_sdss']
df.loc[:,'ri_sdss'] = df.loc[:,'r_sdss'] - df.loc[:,'i_sdss']
df.loc[:,'iz_sdss'] = df.loc[:,'i_sdss'] - df.loc[:,'z_sdss']
df.loc[:,'gi_sdss'] = df.loc[:,'g_sdss'] - df.loc[:,'i_sdss']


In [ ]:
# Insert dmag column...
df.loc[:,'dmag'] = -9999.

In [ ]:
df

## 9. Create Initial Mask

In [ ]:
mask1 = (df["u_sdss"] > 15.) & (df["u_sdss"] < 30.)
mask2 = (df["g_sdss"] > 15.) & (df["g_sdss"] < 30.)
mask3 = (df["r_sdss"] > 15.) & (df["r_sdss"] < 30.)
mask4 = (df["i_sdss"] > 15.) & (df["i_sdss"] < 30.)
mask5 = (df["z_sdss"] > 15.) & (df["z_sdss"] < 30.)
mask6 = df["u_err_sdss"] <= 0.10
mask7 = df["g_err_sdss"] <= 0.05
mask8 = df["r_err_sdss"] <= 0.05
mask9 = df["i_err_sdss"] <= 0.05
mask10 = df["z_err_sdss"] <= 0.10
mask_sdss = mask1 & mask2 & mask3 & mask4 & mask5 & mask6 & mask7 & mask8 & mask9 & mask10

mask1 = (df["g_LSST"] > 16.5) & (df["g_LSST"] < 30.)
mask2 = (df["r_LSST"] > 16.5) & (df["r_LSST"] < 30.)
mask3 = (df["i_LSST"] > 16.5) & (df["i_LSST"] < 30.)
mask4 = (df["z_LSST"] > 16.5) & (df["z_LSST"] < 30.)
mask5 = (df["y_LSST"] > 16.0) & (df["y_LSST"] < 30.)
mask6 = df["g_err_LSST"] <= 0.05
mask7 = df["r_err_LSST"] <= 0.05
mask8 = df["i_err_LSST"] <= 0.05
mask9 = df["z_err_LSST"] <= 0.05
mask10 = df["y_err_LSST"] <= 0.10

# No LSST y-band for overlapping SDSS!
mask_LSST = mask1 & mask2 & mask3 & mask4 & mask5 & mask6 & mask7 & mask8 & mask9 & mask10
#mask_LSST = mask1 & mask2 & mask3 & mask4 & mask6 & mask7 & mask8 & mask9
# No LSST y-band and little LSST z-band for overlapping SDSS!
#mask_LSST = mask1 & mask2 & mask3 & mask6 & mask7 & mask8


mask = mask_sdss & mask_LSST
#mask = mask_LSST


## 10. Make Backup Copies of Initial Mask and Original Data Frame

In [ ]:
# Make a backup copy of original df...
df_orig = df.copy()

# Make a backup copy of original mask...
mask_orig = mask.copy()

## 11. Run Fit in Each Filter Band

In [ ]:
for band in bandList:

    print("")
    print("")
    print("")
    print("# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # ")
    print(band)
    print("# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # ")
    print("")

    magName = band
    magName_match = matchBand_dict[band]
    color_name_1 = color_name_1_dict[band]
    colorLabel_1 = colorLabel_1_dict[band]
    
    # Create title/names for use in QA plots...
    title = """%s --> %s""" % (magName_match, magName)
    dmagName = """%s - %s""" % (magName, magName_match)

    # Grab the original version of df from the backup copy...
    df = df_orig.copy()

    # Grab the original version of mask from the backup copy...
    mask = mask_orig.copy()

    # Update dmag column for {$band} - {$band}_match...
    df.loc[:,'dmag'] = df.loc[:,magName] - df.loc[:,magName_match]

    # Update mask...
    mask1 = ( abs(df['dmag']) <= 10. )
    mask2 = ( ( df[color_name_1] > -3. ) & ( df[color_name_1] < 6.0 ) & (df[color_name_1].notna()) )
    #mask3 = ( ( df[magName] >= mag_limits_dict[magName][0] ) & ( df[magName] <= mag_limits_dict[magName][1] ) & (df[magName].notna()) )
    mask = mask & mask1 & mask2
    #mask = mask & mask1 & mask2 & mask3

    # Apply the new mask to df...
    df = df[mask]

    ## Sanity check
    #df.plot(color_name_1, 'dmag', kind='scatter')

    # Calculate number of disjoint branches to fit...
    nbranches = len(color_limits_1_dict[band]) - 1
    if verbose > 0: print(band, color_limits_1_dict[band], nbranches)

    
    # Reset bluest color limit in bluest branch to color1_min
    #  and reddest color limit in reddest branch to color1_max, 
    #  after excluding above mask...
    #dftmp = df[mask]
    dftmp = df
    color1_desc = dftmp[color_name_1].describe(percentiles=[0.0001, 0.9999])
    color1_min = math.floor(10*(color1_desc['0.01%']-0.05))/10.
    color1_max = math.ceil(10*(color1_desc['99.99%']+0.05))/10.
    color_limits_1_dict[band][0] = color1_min
    color_limits_1_dict[band][nbranches] = color1_max
    if verbose > 0: print(band, color_limits_1_dict[band], nbranches)

    p_branch_list = []
    
    # Iterate, with sigma-clipping...
    df_list = []
    color1_list = []
    dmag_list = []
    res_list = []
    mask_list = []
    plot1Text = ''
    outputLine = ''
        
    for ibranch in range(nbranches):
            
        print('*********************************')
        print(ibranch, color_limits_1_dict[band][ibranch], color_limits_1_dict[band][ibranch+1])
        print('*********************************')

        # Extract branch...
        mask_branch = ( (df[color_name_1] >  color_limits_1_dict[band][ibranch]) & \
                        (df[color_name_1] <= color_limits_1_dict[band][ibranch+1]) )

        df_branch = df[mask_branch].copy()

        # ... and extract dmag's and color1's for this branch...
        dmag_branch   = df_branch.loc[:,'dmag']
        color1_branch = df_branch.loc[:,color_name_1]

        # If there are no valid colors on this branch, continue to next branch...
        if len(color1_branch) < 1:
            continue
                
        # Perform fit for each disjoint branch...
        print("mask_branch length (before): " , len(mask_branch))
        print("dmag_branch length:  ", len(dmag_branch))
        print("color1_branch length:  ", len(color1_branch))
        p_branch, perr_branch, color1_branch, dmag_branch, res_branch, stddev_branch = \
                                poly_fit_with_sigma_clip(color1_branch, dmag_branch, degree=norder)
        print("mask_branch length (after): " , len(mask_branch))

        # NEW:  13 June 2026:
        # Update branch color limits based on clipped data
        lo_new = np.min(color1_branch)
        hi_new = np.max(color1_branch)
        # Store updated limits
        color_limits_1_dict[band][ibranch] = lo_new
        color_limits_1_dict[band][ibranch+1] = hi_new


        # Print coefficients and estimated statistical errors in the coefficients
        for i, (p, perr) in enumerate(zip(p_branch, perr_branch)):
            print(f'p_{len(p_branch)-i-1} = {p:.6f} ± {perr:.6f}')
      
        # Prepare some text output for plots...
        #  Recall that np.polyfit returns the coefficients from highest order to lowest
        #  (This is opposite of the order the coefficients in older versions of this notebook
        #   that did not use np.polyfit for the polynomial fits)
        if norder == 1:
            plot1Text1 = """%s = %.3f + %.3f*%s [%.1f < %s <= %.1f] [rms: %.3f]""" % \
                (dmagName, p_branch[1], p_branch[0], colorLabel_1, \
                 color_limits_1_dict[band][ibranch], colorLabel_1, color_limits_1_dict[band][ibranch+1], \
                 stddev_branch)
        elif norder == 2:
            plot1Text1 = """%s = %.3f + %.3f*%s + %.3f*%s^2  [%.1f < %s <= %.1f] [rms: %.3f]""" % \
                (dmagName, p_branch[2], p_branch[1], colorLabel_1, p_branch[0], colorLabel_1, \
                 color_limits_1_dict[band][ibranch], colorLabel_1, color_limits_1_dict[band][ibranch+1], \
                 stddev_branch)
        else:
            plot1Text1 = ''
            
        plot1Text = """%s\n%s""" % (plot1Text, plot1Text1)
        
        print(plot1Text1)                        
 
            
        # Append branch df and mask to the df_list and mask_list lists, respectively...
        #df_list.append(df_branch.copy())
        #mask_list.append(mask_branch.copy())
        color1_list.append(color1_branch.copy())
        dmag_list.append(dmag_branch.copy())
        res_list.append(res_branch.copy())
        mask_list.append(mask_branch.copy())
        
    # Concatenate the color1, dmag, res, and mask lists for all the branches...
    color1 = pd.Series(np.concatenate(color1_list))
    dmag = pd.Series(np.concatenate(dmag_list))
    res = pd.Series(np.concatenate(res_list))
    mask = pd.Series(np.concatenate(mask_list))
    
    # Calculate the standard deviation for the full piecewise fit...
    stddev = res.std()


    # Output best fits to screen...
    if verbose > 0:
        print("")
        print(title)
        print(plot1Text)
        print("")
    
    # Create QA plots...
    #res =  df.loc[:,'res']
    #dmag =  df.loc[:,'dmag']
    #color1 = df.loc[:,color_name_1]
    #stddev = df['res'].std()
    outputFileName = """%s.dmag_%s-%s.%s.norder%d.qa1.png""" % \
        (qaFileBaseName, magName, magName_match, color_name_1, norder)
    status = transform1ColorQAPlots1a(dmag, color1, res, norder, title, plot1Text, 
                                 dmagName, colorLabel_1, stddev, outputFileName)  
    
            



In [ ]:
raise StopExecution

## 12.  Sandbox

***ANOTHER METHOD OF QUERYING SDSS DR18*** 

Ideally, we would query the SDSS CasJobs directly from this current Jupyter notebook via the SciScript-Python library ( https://github.com/sciserver/SciScript-Python ), following methodology described in this example SciServer Jupyter notebook:  https://github.com/sciserver/Example-Notebooks/blob/main/SciServer%20Components%20-%20Python%20Examples/CasJobs.ipynb


Here, we merely queried SDSS CasJobs directly using this query:

```
SELECT  
  dbo.fIAUFromEq(s.ra, s.dec) as name, 
  s.ra,s.dec,
  s.psfMag_u,s.psfMag_g,s.psfMag_r,s.psfMag_i,s.psfMag_z,
  s.psfMagErr_u,s.psfMagErr_g,s.psfMagErr_r,s.psfMagErr_i,s.psfMagErr_z,
  r.run,r.stripe  
INTO mydb.LSST_DP1_AREAS_2    
FROM Star s, Run r
WHERE
  s.run = r.run
  AND ( (s.ra BETWEEN 147.0 AND  153.0 AND s.dec BETWEEN -1.0 AND 5.0) )
  AND ((s.flags_u & 0x10000000) != 0) AND ((s.flags_g & 0x10000000) != 0) AND ((s.flags_r & 0x10000000) != 0) AND ((s.flags_i & 0x10000000) != 0) AND ((s.flags_z & 0x10000000) != 0) 
  AND ((s.flags_u & 0x8100000c00a4) = 0) AND ((s.flags_g & 0x8100000c00a4) = 0) AND ((s.flags_r & 0x8100000c00a4) = 0) AND ((s.flags_i & 0x8100000c00a4) = 0) AND ((s.flags_z & 0x8100000c00a4) = 0)     
  AND (((s.flags_u & 0x400000000000) = 0) or (s.psfmagerr_u <= 0.2)) AND (((s.flags_g & 0x400000000000) = 0) or (s.psfmagerr_g <= 0.2)) AND (((s.flags_r & 0x400000000000) = 0) or (s.psfmagerr_r <= 0.2)) 
  AND (((s.flags_i & 0x400000000000) = 0) or (s.psfmagerr_i <= 0.2)) AND (((s.flags_z & 0x400000000000) = 0) or (s.psfmagerr_z <= 0.2)) 
  AND (((s.flags_u & 0x100000000000) = 0) or (s.flags_u & 0x1000) = 0) AND (((s.flags_g & 0x100000000000) = 0) or (s.flags_g & 0x1000) = 0) AND (((s.flags_r & 0x100000000000) = 0) or (s.flags_r & 0x1000) = 0) 
  AND (((s.flags_i & 0x100000000000) = 0) or (s.flags_i & 0x1000) = 0) AND (((s.flags_z & 0x100000000000) = 0) or (s.flags_z & 0x1000) = 0)
```

and downloaded the result as `LSST_COSMOS_dtucker.csv`.